# Baku Sentinel — Flood Risk Forecasting
## End-to-End Pipeline: Ingest → ETL → EDA → Train → Evaluate → Forecast

**Architecture:** Bronze → Silver → Gold medallion (DuckDB) · 6-hourly granularity  
**Model:** XGBoost + isotonic calibration · chronological split · SHAP explainability  
**Forecast:** 15-day live Open-Meteo data · per-zone risk scores & alerts

## Setup

In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib
import duckdb
import shap
from scipy import stats
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
)

# Local project modules
sys.path.append(os.path.abspath('../'))
from src import config, ingestion, pipeline, model as mdl

# Settings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1 · Historical Data Ingestion (Bronze Layer)

This step fetches hourly weather and daily river discharge data for the Baku zones via the Open-Meteo APIs. Data is stored in the Bronze layer of DuckDB.

In [ ]:
ingestion.run_historical_ingest(
    start=config.HISTORICAL_START,
    end=config.HISTORICAL_END,
)

# Verify bronze tables
with duckdb.connect(str(config.DB_PATH)) as conn:
    tables_df = conn.execute("SHOW TABLES").df()

    # Simple check for bronze layer tables
    bronze_tables = conn.execute("""
        SELECT schema, name, column_names
        FROM (SHOW ALL TABLES)
        WHERE schema = 'bronze'
    """).df()

if not bronze_tables.empty:
    print(f"Bronze tables found: {len(bronze_tables)}")
    print(bronze_tables)
else:
    print("No bronze tables found.")

## 2 · ETL Pipeline — Silver & Gold Layers

This pipeline transforms raw hourly data into a structured 6-hourly format (Silver) and then calculates all hydrological features (Gold) for the model.

In [ ]:
pipeline.run_pipeline()

import duckdb as _ddb

conn = _ddb.connect(str(config.DB_PATH), read_only=True)
gold_exists = conn.execute("""
    SELECT count(*) FROM information_schema.tables
    WHERE table_schema = 'gold' AND table_name = 'flood_features'
""").fetchone()[0]
conn.close()

if not gold_exists:
    print("Gold table does not exist yet.")
    raise SystemExit("Stop here — run ETL first.")

conn = _ddb.connect(str(config.DB_PATH), read_only=True)
df_gold = conn.execute("SELECT * FROM gold.flood_features").df()
conn.close()

df_gold['time_6h'] = pd.to_datetime(df_gold['time_6h'])
print(f"Gold table  : {df_gold.shape[0]:,} rows × {df_gold.shape[1]} columns")
print(f"Date range  : {df_gold['time_6h'].min().date()} → {df_gold['time_6h'].max().date()}")
print(f"Zones       : {df_gold['zone'].unique().tolist()}")
print(f"Flood rate  : {df_gold['is_flood'].mean()*100:.3f}%")
print(f"Columns     : {df_gold.columns.tolist()}")
df_gold.head(3)


## 3 · Exploratory Data Analysis

### 3.1 · Missing values & data quality

In [ ]:
# Check for missing values
missing_data = df_gold.isnull().mean().sort_values(ascending=False)
missing_only = missing_data[missing_data > 0]

# Visualize missingness
plt.figure(figsize=(10, 4))
if len(missing_only) > 0:
    missing_only.plot(kind='barh', color='salmon')
    plt.title('Missing Value Rate per Feature')
    plt.xlabel('Fraction of Missing Data')
else:
    plt.text(0.5, 0.5, 'No missing values found', ha='center', va='center')
    plt.title('Data Quality: 100% Complete')

plt.tight_layout()
plt.show()

print(f"Total NaN count: {df_gold.isnull().sum().sum()}")

### 3.2 · Flood event distribution by zone & season

In [ ]:
# Create a 3-panel EDA dashboard
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Spatial Analysis: Flood rate by zone
(df_gold.groupby('zone')['is_flood'].mean() * 100).sort_values().plot(
    kind='barh', ax=axes[0], color='skyblue', edgecolor='black'
)
axes[0].set_title('Flood Prevalence by Zone')
axes[0].set_xlabel('Probability (%)')

# 2. Seasonal Analysis: Seasonality of flood events
# Using dt.month directly for grouping
monthly_stats = df_gold.pivot_table(index=df_gold['time_6h'].dt.month,
                                   columns='zone',
                                   values='is_flood',
                                   aggfunc='mean')
monthly_stats.plot(ax=axes[1], marker='o', alpha=0.8)
axes[1].set_title('Seasonal Trends')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

# 3. Temporal Analysis: Low Relief timeline
low_relief = df_gold[df_gold['zone'] == 'Low Relief'].set_index('time_6h')
low_relief['is_flood'].resample('ME').mean().plot(
    ax=axes[2], color='indianred', linewidth=2
)
axes[2].set_title('Low Relief: Long-term Trend')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

### 3.3 · Correlation analysis — Spearman ρ with is_flood

In [ ]:
# Identify numerical features and target
num_cols = df_gold.select_dtypes(include='number').columns.drop('is_flood').tolist()
target = 'is_flood'

# Calculate Spearman correlation for each feature
correlations = []
for col in num_cols:
    # Drop NaNs to ensure clean calculation
    valid_data = df_gold[[col, target]].dropna()

    if len(valid_data) > 50:
        rho, pval = stats.spearmanr(valid_data[col], valid_data[target])
        correlations.append({'feature': col, 'rho': rho, 'pval': pval})

# Create correlation dataframe and take top 25
corr_df = pd.DataFrame(correlations)
corr_df['abs_rho'] = corr_df['rho'].abs()
top_features = corr_df.sort_values('abs_rho', ascending=False).head(25)

# Visualize Top 25 correlations
plt.figure(figsize=(10, 8))
colors = ['indianred' if r > 0 else 'steelblue' for r in top_features['rho']]

plt.barh(top_features['feature'], top_features['rho'], color=colors, alpha=0.9)
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')

plt.title('Feature Importance: Spearman Correlation with Flood Events')
plt.xlabel('Spearman Rho (ρ)')
plt.gca().invert_yaxis() # Highest correlation at the top
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Quick insights
print("Strongest Positive Drivers:")
print(top_features[top_features['rho'] > 0][['feature', 'rho']].head(10))

### 3.4 · Multicollinearity Analysis: Identifying Redundant Features

In [ ]:
# 1. Calculate correlation for all numerical features
all_num_features = df_gold.select_dtypes(include='number').columns.drop('is_flood')
full_corr_matrix = df_gold[all_num_features].corr(method='spearman')

# 2. Setup the visualization with forced labels
plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(full_corr_matrix, dtype=bool))

sns.heatmap(full_corr_matrix,
            mask=mask,
            annot=False,
            cmap='RdBu_r',
            center=0,
            linewidths=0.05,
            square=True,
            cbar_kws={"shrink": .7},
            xticklabels=True,
            yticklabels=True)

plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7)
plt.title('Global Multicollinearity Matrix (All Features)', fontsize=15)
plt.show()

# 3. Extract and display redundant pairs (>0.90)
upper_tri = full_corr_matrix.where(np.triu(np.ones(full_corr_matrix.shape), k=1).astype(bool))
redundant_pairs = upper_tri.stack().sort_values(ascending=False)

print("Top Redundant Feature Pairs (>0.90):")
print("-" * 40)
print(redundant_pairs[redundant_pairs > 0.90])

In [ ]:
# Definitive list of redundant or non-predictive features to exclude
exclude_list = [
    'humidity_precip_product',   # Proxy for raw precipitation
    'precip_roll_max_24h',       # Redundant with precip_roll_sum_24h
    'zone_cascade_risk',         # Redundant with highland_precip_24h
    'soil_temperature_0_to_7cm', # Redundant with air temperature_2m
    'soil_saturation_index',     # Redundant with soil_moisture_0_to_7cm
    'river_discharge',           # Using lags instead to ensure predictive power
    'discharge_roll_max_24h',    # Redundant with discharge_lag_6h
    'temp_lag_24h',              # Redundant with temperature_2m
    'discharge_lag_24h',         # Redundant with discharge_lag_6h
    'precip_roll_sum_24h'        # Redundant with highland_precip_24h
]

# Create the final feature list from all available numerical columns
final_features = [f for f in all_num_features if f not in exclude_list]

print(f"Features filtered: {len(all_num_features)} → {len(final_features)}")
print("-" * 30)
print("Final Model Input Features:")
print(final_features)

# Create the model-ready dataframe
# We keep 'is_flood' and 'time_6h' for splitting and evaluation logic
df_model = df_gold[final_features + ['is_flood', 'time_6h']].copy()

print("-" * 30)
print(f"Cleanup complete: {len(exclude_list)} redundant features removed.")
print(f"Total features retained: {len(final_features)}")

### 3.5 · Mann-Whitney U test — flood vs. non-flood distributions

In [ ]:
# Separate classes to compare distributions
flood_samples = df_gold[df_gold['is_flood'] == 1]
normal_samples = df_gold[df_gold['is_flood'] == 0]

comparison_results = []

# Analyze every feature in the final set
for col in final_features:
    f_data = flood_samples[col].dropna()
    nf_data = normal_samples[col].dropna()

    # Statistically sound check: minimum 15 samples per class
    if len(f_data) > 15 and len(nf_data) > 15:
        stat, pval = stats.mannwhitneyu(f_data, nf_data, alternative='two-sided')

        comparison_results.append({
            'Feature': col,
            'Flood Median': f_data.median(),
            'Normal Median': nf_data.median(),
            'Difference': f_data.median() - nf_data.median(),
            'p_value': pval
        })

# Store all results, sorted by the most significant (lowest p-value)
stats_df = pd.DataFrame(comparison_results).sort_values('p_value')

# Visualization: Show Top 20 to maintain readability
plt.figure(figsize=(9, 7))
top_stats = stats_df.head(20)

plt.barh(top_stats['Feature'],
         -np.log10(top_stats['p_value'] + 1e-300),
         color='mediumseagreen')

plt.axvline(-np.log10(0.05), color='indianred', linestyle='--', label='Alpha = 0.05')

plt.title('Top 20 Statistical Drivers of Flood Events', fontsize=14)
plt.xlabel('Significance Score (-log10 p-value)')
plt.gca().invert_yaxis()
plt.legend()
plt.tight_layout()
plt.show()

# Print summary for the entire feature set
significant_count = sum(stats_df['p_value'] < 0.05)
print(f"Statistical Summary: {significant_count} / {len(stats_df)} features are significant (p < 0.05)")

# Show the Top 10 strongest features numerically
display(stats_df[['Feature', 'Flood Median', 'Normal Median', 'Difference', 'p_value']].head(10))

In [ ]:
# Identify features that are not statistically significant (p >= 0.05)
insignificant_features = stats_df[stats_df['p_value'] >= 0.05]['Feature'].tolist()

# Create the final list by keeping only significant, non-redundant features
model_ready_features = [f for f in final_features if f not in insignificant_features]

print(f"Initial candidates: {len(final_features)}")
print(f"Dropped for lack of significance: {len(insignificant_features)} ({insignificant_features})")
print(f"Final features for df_model: {len(model_ready_features)}")

In [ ]:
# Rebuild the final dataframe
# We keep the features, the target, and the time for splitting
df_model = df_gold[model_ready_features + ['is_flood', 'time_6h']].copy()

print("-" * 30)
print("Final Features Retained:")
print(model_ready_features)

### 3.6 · Feature distributions — flood vs. non-flood

In [ ]:
# 1. Identify the top 9 most significant features from our cleaned model list
# This ensures we only plot features we are actually keeping for the model
model_ready_stats = stats_df[stats_df['Feature'].isin(model_ready_features)]
plot_features = model_ready_stats['Feature'].head(9).tolist()

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feat in enumerate(plot_features):
    ax = axes[i]

    f_vals = flood_samples[feat].dropna()
    n_vals = normal_samples[feat].dropna()

    # Plotting distributions
    sns.kdeplot(n_vals, ax=ax, fill=True, color='steelblue', label='Normal', alpha=0.4)
    sns.kdeplot(f_vals, ax=ax, fill=True, color='indianred', label='Flood', alpha=0.6)

    # Adding vertical lines for medians to show the 'gap'
    ax.axvline(n_vals.median(), color='steelblue', linestyle='--', alpha=0.8)
    ax.axvline(f_vals.median(), color='indianred', linestyle='--', alpha=0.8)

    ax.set_title(f"{feat}", fontsize=11, fontweight='bold')
    ax.set_ylabel('Density')
    ax.set_xlabel('')

    if i == 0:
        ax.legend(frameon=True)

plt.suptitle('Physical Separation: Key Drivers for the Baku Flood Model', fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

### 3.7 · Correlation heatmap — top engineered features

In [ ]:
# We include 'is_flood' just to see how the final features correlate with the target
final_audit_features = model_ready_features + ['is_flood']

# Create the visualization
fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df_gold[final_audit_features].corr(method='spearman')

# Mask the upper triangle for a cleaner look
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    annot_kws={'size': 7},
    linewidths=0.3
)

ax.set_title('Final Audit: Spearman Correlation of Model-Ready Features')
plt.tight_layout()
plt.show()

## 4 · Model Training

XGBoost classifier with:
- Chronological train/test split (last 20% of time = test)
- `scale_pos_weight` to handle class imbalance
- Isotonic calibration for reliable probabilities

In [ ]:
bundle = mdl.train(df_gold, calibrate=True)

print("\n── Model Metrics ──────────────────────────────────────────")
for k, v in bundle['metrics'].items():
    if k not in ('report', 'cv_auc_pr_mean', 'cv_auc_pr_std'):
        print(f"  {k:25s}: {v}")
print(f"  {'cv_auc_pr':25s}: {bundle['metrics']['cv_auc_pr_mean']:.4f} ± {bundle['metrics']['cv_auc_pr_std']:.4f}")

In [ ]:
# Print the full list of features used
print(f"Total features used: {len(bundle['features'])}")
print("-" * 30)
for feature in bundle['features']:
    print(f" - {feature}")

## 5 · Model Evaluation

### 5.1 · ROC, Precision-Recall curves & Confusion Matrix

In [ ]:
test_df   = bundle['test_df']
y_test    = test_df['is_flood'].values
y_prob    = test_df['risk_score'].values
y_pred    = test_df['flood_pred'].values
threshold = bundle['threshold']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name='Baku Sentinel', color='darkorange')
axes[0].plot([0,1],[0,1],'k--',lw=0.8)
axes[0].set_title(f"ROC Curve (AUC: {roc_auc_score(y_test, y_prob):.3f})")

# 2. Precision-Recall Curve (The most important one for floods)
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name='Baku Sentinel', color='teal')
# Add a baseline for random guessing (fraction of positives)
baseline = y_test.sum() / len(y_test)
axes[1].axhline(baseline, color='gray', linestyle='--', label=f'Baseline ({baseline:.2f})')
axes[1].set_title(f"PR Curve (AUC-PR: {average_precision_score(y_test, y_prob):.3f})")
axes[1].legend()

# 3. Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[2],
                                         colorbar=False, cmap='Blues',
                                         display_labels=['No Flood', 'Flood'])
axes[2].set_title(f"Confusion Matrix (T = {threshold:.3f})")

plt.tight_layout()
plt.show()

# 4. Detailed Metrics
print("\n── Classification Report ──────────────────────────────────")
print(classification_report(y_test, y_pred, target_names=['No Flood','Flood'], zero_division=0))

### 5.2 · Calibration curve

In [ ]:
from sklearn.calibration import calibration_curve

# Using 'quantile' to ensure every point has a statistically significant number of samples
fraction_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy='quantile')

fig, ax = plt.subplots(figsize=(7, 6))

# Perfect calibration line
ax.plot([0,1], [0,1], 'k--', lw=1, alpha=0.6, label='Perfectly Calibrated')

# Our model's performance
ax.plot(mean_pred, fraction_pos, 'o-', color='#185FA5', lw=2,
        label='Baku Sentinel (Isotonic)', markersize=7)

# Visual polish
ax.set_xlabel('Predicted Probability (Risk Score)')
ax.set_ylabel('Observed Frequency (Actual Floods)')
ax.set_title('Reliability Diagram: Probability vs. Reality')
ax.grid(True, alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()

### 5.3 · Threshold sensitivity — F1, Precision, Recall vs. threshold

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_arr, recall_arr, thresholds_arr = precision_recall_curve(y_test, y_prob)
f1_arr = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-9)

# Find values at the chosen threshold (0.30)
idx = np.argmin(np.abs(thresholds_arr - threshold))
p_at_t = precision_arr[idx]
r_at_t = recall_arr[idx]

fig, ax = plt.subplots(figsize=(10, 5))

# Plot the three curves
ax.plot(thresholds_arr, precision_arr[:-1], label='Precision', color='#185FA5', lw=1.5)
ax.plot(thresholds_arr, recall_arr[:-1],    label='Recall',    color='#1D9E75', lw=1.5)
ax.plot(thresholds_arr, f1_arr[:-1],        label='F1 Score',  color='#E24B4A', lw=2)

# Mark the chosen threshold (0.30)
ax.axvline(threshold, color='black', linestyle='--', lw=1.2, label=f'Chosen threshold ({threshold:.2f})')
ax.scatter([threshold], [p_at_t], color='#185FA5', edgecolors='white', zorder=5)
ax.scatter([threshold], [r_at_t], color='#1D9E75', edgecolors='white', zorder=5)

# Visual Polish
ax.set_xlabel('Probability Threshold')
ax.set_ylabel('Score')
ax.set_title('Baku Sentinel: Threshold Trade-off Analysis', fontsize=13)
ax.legend(frameon=True, loc='lower center', ncol=4)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.15)

# Annotation to explain the choice
ax.annotate(f'Recall: {r_at_t:.2f}\nPrec: {p_at_t:.2f}',
            xy=(threshold, r_at_t), xytext=(threshold+0.05, r_at_t+0.1),
            arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

### 5.4 · Risk score distribution by true label

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(test_df[test_df['is_flood']==0]['risk_score'], bins=60,
        alpha=0.5, color='#378ADD', density=True, label='No flood')
ax.hist(test_df[test_df['is_flood']==1]['risk_score'], bins=60,
        alpha=0.6, color='#E24B4A', density=True, label='Flood')
ax.axvline(threshold, color='black', linestyle='--', lw=1.2,
           label=f'Threshold ({threshold:.3f})')
ax.set_xlabel('Predicted flood probability')
ax.set_ylabel('Density')
ax.set_title('Risk score distribution by true label')
ax.legend()
plt.tight_layout()
plt.show()

### 5.5 · Temporal error analysis — risk score over test period

In [ ]:
fig, axes = plt.subplots(len(config.BAKU_ZONES), 1,
                          figsize=(14, 4 * len(config.BAKU_ZONES)), sharex=True)

for ax, zone_cfg in zip(axes, config.BAKU_ZONES):
    zone = zone_cfg['zone']
    zdf  = test_df[test_df['zone'] == zone].sort_values('time_6h')
    ax.fill_between(zdf['time_6h'], zdf['risk_score'], alpha=0.3, color='#378ADD')
    ax.plot(zdf['time_6h'], zdf['risk_score'], lw=0.8, color='#185FA5')
    floods = zdf[zdf['is_flood'] == 1]
    ax.scatter(floods['time_6h'], floods['risk_score'],
               color='#E24B4A', s=12, zorder=5, label='True flood event')
    ax.axhline(threshold, color='black', linestyle='--', lw=0.8)
    ax.axhline(config.RISK_HIGH,   color='#E24B4A', linestyle=':', lw=0.8, alpha=0.7)
    ax.axhline(config.RISK_MEDIUM, color='#EF9F27', linestyle=':', lw=0.8, alpha=0.7)
    ax.set_ylabel('Risk score')
    ax.set_title(f'{zone}')
    ax.set_ylim(0, 1)
    if floods.shape[0] > 0:
        ax.legend(fontsize=8)

axes[-1].set_xlabel('Date')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30)
plt.suptitle('Test period: flood risk score vs. true events', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 6 · SHAP Feature Importance & Explainability

In [ ]:
# Rebuild SHAP values for the test set
checkpoint = mdl.load_model()
feature_cols = checkpoint['features']

drop_cols = ['time_6h', 'river_discharge', 'is_flood',
             'risk_score', 'flood_pred', 'risk_level']
X_test_raw = pd.get_dummies(
    test_df.drop(columns=[c for c in drop_cols if c in test_df.columns]),
    columns=['zone'], drop_first=True
)
for c in feature_cols:
    if c not in X_test_raw.columns:
        X_test_raw[c] = 0
X_test_enc = X_test_raw[feature_cols].fillna(0)

# Get the base XGBoost estimator for SHAP
base_model = checkpoint['model']
if hasattr(base_model, 'estimator'):          # CalibratedClassifierCV
    base_model = base_model.estimator

explainer   = shap.TreeExplainer(base_model)
shap_values = explainer.shap_values(X_test_enc)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

print(f"SHAP values computed for {X_test_enc.shape[0]:,} test samples.")

### 6.1 · Global feature importance (mean |SHAP|)

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_ser = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
shap_ser.sort_values().plot(kind='barh', ax=ax, color='#534AB7', alpha=0.85)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Top 20 features by global SHAP importance')
plt.tight_layout()
plt.show()

print("Top 15 features:")
print(shap_ser.head(15).to_string())

### 6.2 · SHAP summary plot (beeswarm)

In [ ]:
shap.summary_plot(shap_values, X_test_enc,
                  feature_names=feature_cols,
                  max_display=20,
                  show=True)

### 6.3 · SHAP dependence plots — top 4 features

In [ ]:
top4 = shap_ser.index[:4].tolist()
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, feat in zip(axes.flatten(), top4):
    if feat not in X_test_enc.columns:
        continue
    idx = feature_cols.index(feat)
    ax.scatter(X_test_enc[feat], shap_values[:, idx],
               alpha=0.3, s=8, c=shap_values[:, idx],
               cmap='RdBu_r')
    ax.axhline(0, color='gray', lw=0.6)
    ax.set_xlabel(feat)
    ax.set_ylabel('SHAP value')
    ax.set_title(f'Dependence: {feat}')

plt.suptitle('SHAP dependence plots — top 4 features', y=1.01)
plt.tight_layout()
plt.show()

## 7 · Live 15-Day Flood Risk Forecast

Fetches real-time data from Open-Meteo, processes through the Sentinel pipeline,
and scores with the trained model.

In [ ]:
# 7.1 Fetch live forecast
df_forecast_raw = ingestion.fetch_all_forecast(forecast_days=config.FORECAST_DAYS)
print(f"Live forecast: {len(df_forecast_raw):,} hourly rows across "
      f"{df_forecast_raw['zone'].nunique()} zones")
df_forecast_raw.head(3)

In [ ]:
# 7.2 Run through Sentinel pipeline (Silver stream → Gold features)
df_features = pipeline.run_forecast_pipeline(df_forecast_raw)
print(f"Engineered features: {df_features.shape}")
df_features.head(3)

In [ ]:
import importlib
import src.model as mdl

# Force Python to read the new version of model.py
importlib.reload(mdl)

In [ ]:
# 7.3 Score with trained model
checkpoint   = mdl.load_model()
df_predicted = mdl.predict(df_features, checkpoint=checkpoint)

# Summary alert table
summary = (df_predicted
    .groupby('zone')
    .agg(
        max_risk  =('risk_score', 'max'),
        mean_risk =('risk_score', 'mean'),
        high_risk_periods=('risk_score', lambda x: (x >= config.RISK_HIGH).sum()),
        med_risk_periods =('risk_score', lambda x: ((x >= config.RISK_MEDIUM) & (x < config.RISK_HIGH)).sum()),
    )
    .round(3)
    .sort_values('max_risk', ascending=False)
)
print("\n── Zone Risk Summary ─────────────────────────────────────────")
print(summary.to_string())

### 7.4 · Forecast risk score — 15-day time series

In [ ]:
fig, axes = plt.subplots(len(config.BAKU_ZONES), 1,
                          figsize=(14, 4 * len(config.BAKU_ZONES)), sharex=True)

for ax, zone_cfg in zip(axes, config.BAKU_ZONES):
    zone = zone_cfg['zone']
    zdf  = df_predicted[df_predicted['zone'] == zone].sort_values('time_6h')

    ax.fill_between(zdf['time_6h'], zdf['risk_score'],
                    where=zdf['risk_score'] >= config.RISK_HIGH,
                    alpha=0.3, color='#E24B4A', label='HIGH zone')
    ax.fill_between(zdf['time_6h'], zdf['risk_score'],
                    where=(zdf['risk_score'] >= config.RISK_MEDIUM) & (zdf['risk_score'] < config.RISK_HIGH),
                    alpha=0.25, color='#EF9F27', label='MEDIUM zone')
    ax.plot(zdf['time_6h'], zdf['risk_score'], lw=1.2, color='#185FA5')

    ax.axhline(config.RISK_HIGH,   color='#E24B4A', linestyle='--', lw=0.9, alpha=0.8)
    ax.axhline(config.RISK_MEDIUM, color='#EF9F27', linestyle='--', lw=0.9, alpha=0.8)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Flood probability')
    ax.set_title(f'{zone}')
    ax.legend(fontsize=8, loc='upper right')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30)
axes[-1].set_xlabel('Date')
plt.suptitle(f'15-Day Flood Risk Forecast — Baku Sentinel', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

### 7.5 · Alert summary

In [ ]:
ALERT_ICONS = {'HIGH': '⚠️ ', 'MEDIUM': '🟡', 'LOW': '🟢'}

for _, row in summary.iterrows():
    zone      = row.name
    max_risk  = row['max_risk']
    high_p    = int(row['high_risk_periods'])
    if max_risk >= config.RISK_HIGH:
        level = 'HIGH'
    elif max_risk >= config.RISK_MEDIUM:
        level = 'MEDIUM'
    else:
        level = 'LOW'
    icon = ALERT_ICONS[level]
    print(f"{icon}  {zone:20s}  peak={max_risk*100:.1f}%  "
          f"high-risk 6h-periods={high_p}  level={level}")